# LLM Evaluation — From Metrics to Custom Eval Pipelines

**Session 7 — Applied Natural Language Processing**

Good prompting is only half the picture. How do we know if our prompts actually work?

This notebook covers three tiers of LLM evaluation:
- **Tier A — Classical metrics**: BLEU, ROUGE, BERTScore, perplexity
- **Tier B — Eval tools**: Promptfoo for prompt testing
- **Tier C — Custom pipelines**: DIY eval loop and LLM-as-judge

**Part 1** requires no API key. **Part 2** requires a GitHub token (set `GITHUB_TOKEN` env var).

In [1]:
# Part 1 dependencies — no API key needed
# Uncomment to install if not already in your environment:
# !pip install evaluate bert-score transformers torch

import warnings
warnings.filterwarnings("ignore")

import evaluate
from bert_score import score as bert_score
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from dotenv import load_dotenv
import os

load_dotenv()

print("Part 1 dependencies ready.")

Part 1 dependencies ready.


In [4]:
# Part 2 dependencies — requires GITHUB_TOKEN
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.environ.get("GITHUB_TOKEN", "set-your-github-token"),
)
MODEL = "gpt-4o-mini"

def chat(prompt, system=None, temperature=0.3, max_tokens=300):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=MODEL, messages=messages,
        temperature=temperature, max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

print(chat("Hello, world!", system="You are a helpful assistant."))
print("Part 2 dependencies ready.")

Hello! How can I assist you today?
Part 2 dependencies ready.


---

## Part 1 — Classical Metrics (No API Key Required)

Classical NLP metrics predate LLMs but remain widely used. They measure surface-level or semantic similarity between a model's output and a reference (gold-standard) text.

We'll use three summarisation examples to compare the metrics:
- **Good summary**: captures key points, well-phrased
- **Bad summary**: misses key points, poor quality
- **Synonym summary**: correct meaning, different words (tests metric sensitivity to synonyms)

### 1a — BLEU and ROUGE

**BLEU** (Bilingual Evaluation Understudy) measures n-gram precision — how many n-grams in the model output appear in the reference. Designed for machine translation.

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures n-gram recall — how many reference n-grams appear in the model output. Designed for summarisation.

Both suffer from the same limitation: they measure surface overlap, not meaning. A synonym earns zero credit.

In [6]:
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")

# Source article (what we're summarising)
source = (
    "The city council voted unanimously to approve a new public transport initiative. "
    "The plan includes expanding the metro network by 12 new stations, introducing "
    "electric buses on all major routes, and building 50 new cycling lanes. "
    "The project is expected to reduce car usage by 30% over five years."
)

# Reference: what a human would write as the ideal summary
reference = "The council approved a transport plan to expand metro, add electric buses, and build cycling lanes, aiming to cut car use by 30%."

# Three candidate summaries
candidates = {
    "Good summary":    "The city council approved a transport initiative to expand the metro by 12 stations, introduce electric buses, and add cycling lanes, targeting a 30% reduction in car use.",
    "Bad summary":     "The council had a meeting and discussed various city matters including transport and other infrastructure topics.",
    "Synonym summary": "The municipal council unanimously endorsed a transit scheme to broaden the subway network, deploy electric coaches, and create bicycle paths, aiming to decrease vehicle traffic by 30%.",
}

print(f"{'Summary':<20} {'BLEU':>8} {'ROUGE-1':>10} {'ROUGE-L':>10}")
print("-" * 52)

for name, candidate in candidates.items():
    bleu = bleu_metric.compute(predictions=[candidate], references=[[reference]])
    rouge = rouge_metric.compute(predictions=[candidate], references=[reference])
    print(f"{name:<20} {bleu['bleu']:>8.3f} {rouge['rouge1']:>10.3f} {rouge['rougeL']:>10.3f}")

Summary                  BLEU    ROUGE-1    ROUGE-L
----------------------------------------------------
Good summary            0.225      0.706      0.588
Bad summary             0.000      0.256      0.256
Synonym summary         0.133      0.400      0.400


**What to notice:**
- The **good summary** scores well — it uses similar words to the reference.
- The **synonym summary** may score poorly on BLEU despite being semantically correct — it uses different words.
- The **bad summary** scores near zero, as expected.
- This illustrates the core limitation: BLEU and ROUGE reward word overlap, not meaning.

---

### 1b — BERTScore

BERTScore (Zhang et al., 2020) addresses the synonym problem. Instead of counting matching words, it uses contextual embeddings from BERT to compute semantic similarity. A paraphrase scores highly even if it shares no words with the reference.

In [10]:
# Run BERTScore on the same candidates
references_list = [reference] * len(candidates)
candidates_list = list(candidates.values())
candidate_names = list(candidates.keys())

# Compute BERTScore
P, R, F1 = bert_score(candidates_list, references_list, lang="en", verbose=False)

print(f"{'Summary':<20} {'BERTScore F1':>14}")
print("-" * 36)
for name, f1 in zip(candidate_names, F1):
    print(f"{name:<20} {f1.item():>14.3f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Summary                BERTScore F1
------------------------------------
Good summary                  0.965
Bad summary                   0.883
Synonym summary               0.951


**What to notice:**
- The **synonym summary** should now score similarly to (or better than) the good summary — BERTScore rewards semantic equivalence.
- The **bad summary** still scores low, even with BERTScore.
- This shows the value of embedding-based metrics for tasks where paraphrasing is acceptable (e.g., summarisation, translation).

---

### 1c — Perplexity with GPT-2

**Perplexity** measures how surprised a language model is by a piece of text. Lower perplexity = the model finds the text more probable = the text is more fluent.

Perplexity does **not** measure factual accuracy — a fluent lie scores better than an awkward truth.

Perplexity is effectively:

A transformation of average log-likelihood
Interpretable as “how well the model predicts the next token”

Lower perplexity ⇒ higher assigned probability ⇒ better fit to the text.

More Formally: 
- High perplexity = model assigns low probability to the sequence
- Low perplexity = model assigns high probability to the sequence


Perplexity reflects statistical likelihood, not truth. A grammatically smooth false statement can score better than a clumsy true one


We use GPT-2 (a small, open model) as the language model for this demo.

In [9]:
# Load GPT-2 for perplexity computation
print("Loading GPT-2...")
model_name = "gpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
gpt2_model = GPT2LMHeadModel.from_pretrained(model_name)
gpt2_model.eval()
print("GPT-2 loaded.")

def compute_perplexity(text):
    """Compute perplexity of a text using GPT-2."""
    encodings = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = gpt2_model(**encodings, labels=encodings["input_ids"])
    loss = outputs.loss
    return torch.exp(loss).item()

# Compare fluent vs disfluent text
examples = {
    "Fluent": "The restaurant was beautifully decorated and the service was excellent.",
    "Disfluent": "Restaurant the was decorated beautifully excellent service the was and.",
    "Factually wrong but fluent": "The Eiffel Tower is located in London, where it was built in 1889.",
}

print(f"\n{'Text type':<35} {'Perplexity':>12}")
print("-" * 49)
for label, text in examples.items():
    ppl = compute_perplexity(text)
    print(f"{label:<35} {ppl:>12.1f}")

Loading GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 loaded.

Text type                             Perplexity
-------------------------------------------------
Fluent                                      22.2
Disfluent                                 1138.5
Factually wrong but fluent                   9.8


**What to notice:**
- The disfluent sentence has much higher perplexity — GPT-2 finds it improbable.
- The factually wrong but fluent sentence has **low perplexity** — it reads naturally.
- This demonstrates why perplexity alone cannot catch hallucinations.
- Use perplexity to compare fluency across models, not to verify factual accuracy.

---

### 1d — Leaderboard Exploration

No code needed — this section is guided reading.

**Key leaderboards to explore:**

| Leaderboard | What it measures | URL |
|---|---|---|
| HF Open LLM Leaderboard | Open-source model benchmarks (MMLU, HellaSwag, etc.) | [huggingface.co/spaces/open-llm-leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard) |
| LMSYS Chatbot Arena | Human preference (Elo rating from real user votes) | [arena.ai](https://arena.ai) |
| Artificial Analysis | Intelligence + speed + cost + latency | [artificialanalysis.ai](https://artificialanalysis.ai) |

**Questions to ask when reading a leaderboard:**
1. Is the benchmark saturated? (If top models all score 90%+, small differences are noise.)
2. Is this a reasoning model being compared on reasoning benchmarks only?
3. What is *not* reported? (Labs choose which benchmarks to publish.)
4. Does strong benchmark performance predict strong performance on *my* task? (Usually not directly.)

**Key insight:** Leaderboards give you an evidence-based shortlist. They replace uninformed guessing — but they're proxies for your actual use case. The further your task is from standard benchmarks, the more critical your own evaluation becomes.

---

### 1e — End-to-End: Generate a Dataset and Run the Full Pipeline

*(Requires GitHub Token — uses the `chat()` function from the Part 2 setup cell)*

The previous sections covered evaluation building blocks in isolation. Now let’s assemble the full pipeline:

1. Use the LLM to generate a labelled evaluation dataset
2. Save the dataset to a file (reused in section 2d)
3. Run the LLM on every input to collect predictions
4. Score with exact match and report per-class accuracy

This is the minimal pattern for any evaluation project: generate labelled data → run your model → score → report.

In [ ]:
import json
import pathlib

# --- Step 1: Generate a labelled dataset using the LLM ---
# We ask the model to produce 15 diverse examples we can then evaluate against.

generate_prompt = """Create a JSON array of 15 sentiment analysis test cases.
Each object must have exactly two keys:
- "input": a short review or comment (1-2 sentences, varied topics)
- "expected": the correct label — exactly one of "Positive", "Negative", or "Neutral"

Distribution: exactly 5 Positive, 5 Negative, 5 Neutral.
Vary topics: restaurants, hotels, products, movies, online services.
Make 2 examples per sentiment subtly challenging (e.g. polite complaint, faint praise).

Return ONLY a valid JSON array. No explanation, no markdown fences."""

print("Generating evaluation dataset...")
raw = chat(generate_prompt, temperature=0.7, max_tokens=1500)

# Strip markdown fences if the model adds them
cleaned = raw.strip()
if "```" in cleaned:
    parts = cleaned.split("```")
    # Take the part after the first fence
    cleaned = parts[1]
    if cleaned.startswith("json"):
        cleaned = cleaned[4:]
cleaned = cleaned.strip()

eval_dataset = json.loads(cleaned)
print(f"Generated {len(eval_dataset)} examples")

# Save to file — reused in section 2d
DATASET_PATH = pathlib.Path("eval_dataset.json")
with open(DATASET_PATH, "w") as f:
    json.dump(eval_dataset, f, indent=2)
print(f"Saved to {DATASET_PATH}\n")

# Preview
for item in eval_dataset[:4]:
    print(f"  [{item['expected']:8}] {item['input'][:70]}...")

In [ ]:
# --- Step 2: Run the full eval pipeline on the generated dataset ---

EVAL_PROMPT = """Classify the sentiment as exactly one of: Positive, Negative, or Neutral.
Respond with only the label — no explanation.

Text: {text}
Sentiment:"""

def run_prediction(text):
    """Run the LLM on a single input at temperature=0 for deterministic output."""
    return chat(EVAL_PROMPT.format(text=text), temperature=0.0, max_tokens=10)

def score_exact(prediction, expected):
    """Case-insensitive check that the expected label appears in the prediction."""
    return expected.lower() in prediction.lower()

# Run predictions
print("Running predictions...")
eval_results = []
for item in eval_dataset:
    pred = run_prediction(item["input"])
    correct = score_exact(pred, item["expected"])
    eval_results.append({
        "input": item["input"],
        "expected": item["expected"],
        "prediction": pred.strip(),
        "correct": correct,
    })

# Aggregate overall accuracy
accuracy = sum(r["correct"] for r in eval_results) / len(eval_results)
n_correct = sum(r["correct"] for r in eval_results)
print(f"\nOverall accuracy: {accuracy:.1%}  ({n_correct}/{len(eval_results)})\n")

# Per-class breakdown
for label in ["Positive", "Negative", "Neutral"]:
    subset = [r for r in eval_results if r["expected"] == label]
    n = sum(r["correct"] for r in subset)
    print(f"  {label:8}: {n}/{len(subset)}")

# Per-item table
print()
print(f"{''  :>3} {'Expected':>10} {'Predicted':>12}  Input")
print("-" * 76)
for r in eval_results:
    icon = "\u2713" if r["correct"] else "\u2717"
    print(f"{icon:>3} {r['expected']:>10} {r['prediction']:>12}  {r['input'][:48]}...")

**What to notice:**
- **Neutral is typically the hardest class** — LLMs tend to lean positive or negative on ambiguous text.
- Look at the failures: are they clustered around a specific sentiment class or topic?
- This dataset is saved as `eval_dataset.json` — section 2d will pass it to the OpenAI Evals API and run the same evaluation as a managed job, so you can compare results directly.

---

## Part 2 — Build Your Own Eval Pipeline

*(Requires GitHub Token — set `GITHUB_TOKEN` in your environment)*

Classical metrics measure output quality in isolation. But for real projects, you need evaluation tailored to your specific task. This section builds a custom eval pipeline from first principles, then shows how tools like promptfoo make this scalable.

### 2a — Promptfoo

[Promptfoo](https://github.com/promptfoo/promptfoo) is an open-source CLI tool for prompt testing and model comparison. You define test cases in YAML, run them against one or more models, and get a visual comparison report.

It's particularly useful for:
- Comparing the same prompt across different models
- Regression testing when you update a prompt
- AT2-style projects where you need systematic prompt evaluation

**A working example is in this session's `getting-started/` folder** — it tests a translation prompt against a local Ollama model with no API key required.

To run it:
```
# From material/Session 7/getting-started/
npx promptfoo eval
npx promptfoo view
```

The `promptfooconfig.yaml` shows the core pattern: prompts → providers → test cases with assertions.

In [11]:
# Display the promptfoo config from the getting-started example
with open("../getting-started/promptfooconfig.yaml") as f:
    print(f.read())

# yaml-language-server: $schema=https://promptfoo.dev/config-schema.json
description: 'Getting started'
# Optionally set API keys here instead of exporting environment variables.
# Never commit real keys to source control.
# env:
#   OPENAI_API_KEY: sk-...
prompts:
  - "Convert the following English text to {{language}}: {{input}}"

providers:
  - ollama:gpt-oss:latest
  # - openai:gpt-5.2
  # - openai:gpt-5-mini
  # Or setup models from other providers
  # - anthropic:messages:claude-sonnet-4-6
  # - google:gemini-3.1-pro-preview

defaultTest:
  options:
    provider: ollama:gpt-oss:latest
  assert:
    - type: llm-rubric
      value: >-
        Does the response correctly translate the input text into the target language?
        Give it a rating on a scale from 1 to 5, where 1 is a poor translation and 5 is an excellent translation that captures the meaning and nuances of the original text.
tests:
  - vars:
      language: French
      input: Hello world
    assert:
      - type: co

**When to use promptfoo vs. a Python eval loop:**
- Use **promptfoo** when you want to compare multiple models/prompts visually, or need a shareable report.
- Use a **Python eval loop** (below) when you need custom scoring logic, integration with your own data, or fine-grained control over the eval process.

Both patterns are complementary — promptfoo for exploration, Python loops for production eval pipelines.

---

### 2b — DIY Eval Loop

The core pattern for any eval pipeline:
1. Define test cases (input + expected output)
2. Run the model on each input
3. Score each output
4. Aggregate results

Let's build this for a sentiment analysis task — the same task we used in the prompting notebook.

In [12]:
# 8 sentiment test cases: input text + expected label
test_cases = [
    {"input": "I absolutely loved the new restaurant! The food was amazing.", "expected": "Positive"},
    {"input": "The movie was a complete waste of time. Terrible plot.", "expected": "Negative"},
    {"input": "The package arrived on time. Nothing special.", "expected": "Neutral"},
    {"input": "Despite the wait, the food was delicious and staff were friendly.", "expected": "Positive"},
    {"input": "I'm so disappointed. The product arrived damaged and customer service ignored me.", "expected": "Negative"},
    {"input": "The hotel room was clean and functional. Met expectations.", "expected": "Neutral"},
    {"input": "Best coffee I've ever had! I'll definitely come back.", "expected": "Positive"},
    {"input": "Not the worst, but not great either. Average in every way.", "expected": "Neutral"},
]

print(f"Test set: {len(test_cases)} cases")
for i, tc in enumerate(test_cases, 1):
    print(f"  {i}. [{tc['expected']:8}] {tc['input'][:60]}...")

Test set: 8 cases
  1. [Positive] I absolutely loved the new restaurant! The food was amazing....
  2. [Negative] The movie was a complete waste of time. Terrible plot....
  3. [Neutral ] The package arrived on time. Nothing special....
  4. [Positive] Despite the wait, the food was delicious and staff were frie...
  5. [Negative] I'm so disappointed. The product arrived damaged and custome...
  6. [Neutral ] The hotel room was clean and functional. Met expectations....
  7. [Positive] Best coffee I've ever had! I'll definitely come back....
  8. [Neutral ] Not the worst, but not great either. Average in every way....


In [13]:
SENTIMENT_PROMPT = """Classify the sentiment of the following text as exactly one of: Positive, Negative, or Neutral.
Respond with only the label — no explanation.

Text: {text}
Sentiment:"""

In [14]:
def run_model(text):
    """Run the model on a single input and return the output."""
    prompt = SENTIMENT_PROMPT.format(text=text)
    return chat(prompt, temperature=0.0, max_tokens=10)

def exact_match(prediction, expected):
    """Check if the expected label appears in the prediction (case-insensitive)."""
    return expected.lower() in prediction.lower()

In [15]:
import json

results = []
for case in test_cases:
    prediction = run_model(case["input"])
    correct = exact_match(prediction, case["expected"])
    results.append({
        "input": case["input"],
        "expected": case["expected"],
        "prediction": prediction.strip(),
        "correct": correct,
    })

accuracy = sum(r["correct"] for r in results) / len(results)
print(f"Accuracy: {accuracy:.1%}  ({sum(r['correct'] for r in results)}/{len(results)} correct)\n")

for r in results:
    icon = "\u2713" if r["correct"] else "\u2717"
    print(f"[{icon}] Expected: {r['expected']:8} | Got: {r['prediction']:12} | {r['input'][:55]}...")

Accuracy: 87.5%  (7/8 correct)

[✓] Expected: Positive | Got: Positive     | I absolutely loved the new restaurant! The food was ama...
[✓] Expected: Negative | Got: Negative     | The movie was a complete waste of time. Terrible plot....
[✓] Expected: Neutral  | Got: Neutral      | The package arrived on time. Nothing special....
[✓] Expected: Positive | Got: Positive     | Despite the wait, the food was delicious and staff were...
[✓] Expected: Negative | Got: Negative     | I'm so disappointed. The product arrived damaged and cu...
[✗] Expected: Neutral  | Got: Positive     | The hotel room was clean and functional. Met expectatio...
[✓] Expected: Positive | Got: Positive     | Best coffee I've ever had! I'll definitely come back....
[✓] Expected: Neutral  | Got: Neutral      | Not the worst, but not great either. Average in every w...


**What to notice:**
- This is the fundamental eval loop pattern. Everything else is variations of: run → score → aggregate.
- Look at the **failures** — are they systematically wrong on neutral cases? Ambiguous reviews? This tells you where to improve the prompt.
- The prompt uses `temperature=0.0` for consistency — you want deterministic labels for evaluation.
- `exact_match` is a simple scorer. It works for classification. For open-ended tasks, you need something smarter — which is where LLM-as-judge comes in.

---

### 2c — LLM-as-Judge

For open-ended tasks (summarisation, explanation, QA), there's no single correct answer — `exact_match` doesn't work. **LLM-as-judge** uses a strong model to evaluate the output of another model against a rubric.

Pattern:
1. Define a rubric (scoring criteria)
2. Send the question, response, and rubric to a judge model
3. Ask for a score + justification in JSON format
4. Use `temperature=0` for the judge — you want consistent, deterministic scoring

In [16]:
def llm_judge(question, response, rubric, judge_model="gpt-4o-mini"):
    """Use an LLM to score a response against a rubric. Returns (score, justification)."""
    prompt = f"""You are an expert evaluator. Score the following response using the rubric provided.

Question: {question}
Response: {response}
Rubric: {rubric}

Respond with ONLY a JSON object: {{"score": <int 1-5>, "justification": "<2 sentences>"}}"""

    raw = chat(prompt, temperature=0, max_tokens=200)

    # Parse JSON response
    try:
        # Handle cases where the model wraps JSON in markdown
        cleaned = raw.strip().strip("```json").strip("```").strip()
        data = json.loads(cleaned)
        return data["score"], data["justification"]
    except json.JSONDecodeError:
        return None, f"Parse error: {raw}"

In [17]:
# Test the judge on a summarisation task
summarisation_rubric = """
5 - Captures all key points, concise, no hallucinations or omissions
4 - Captures most key points, minor omissions only
3 - Captures some key points, notable gaps or inaccuracies
2 - Mostly misses key points or contains significant errors
1 - Completely wrong, irrelevant, or incoherent
"""

source_article = (
    "Researchers at MIT have developed a new battery technology that charges in under 2 minutes "
    "and holds three times more energy than conventional lithium-ion batteries. The breakthrough "
    "uses a novel solid-state electrolyte and could enable electric vehicles to travel over 1,000 km "
    "on a single charge. Commercial production is expected within 5 years."
)

summaries_to_evaluate = [
    ("Good summary", "MIT researchers created a battery that charges in 2 minutes, holds 3x more energy than lithium-ion, enabling EVs to travel 1,000+ km per charge, with commercial availability in 5 years."),
    ("Partial summary", "MIT researchers made a new battery that charges fast and holds more energy."),
    ("Hallucinated summary", "Stanford researchers developed a solar-powered battery that can charge in 30 seconds and lasts a lifetime. It will be available next year for $99."),
]

question = f"Summarise this article in one sentence: {source_article}"

print(f"{'Summary type':<20} {'Score':>7} Justification")
print("-" * 90)
for name, summary in summaries_to_evaluate:
    score, justification = llm_judge(question, summary, summarisation_rubric)
    print(f"{name:<20} {str(score):>7}  {justification[:70]}...")

Summary type           Score Justification
------------------------------------------------------------------------------------------
Good summary               5  The response effectively captures all key points from the article, inc...
Partial summary            4  The response captures the essence of the new battery technology and it...
Hallucinated summary       1  The response is completely wrong and irrelevant, as it discusses a dif...


**What to notice:**
- LLM-as-judge handles open-ended tasks that `exact_match` cannot.
- The hallucinated summary should score very low — the judge can detect factual errors.


---

### 2d — OpenAI Evals API

The OpenAI Evals API is the managed version of the DIY pipeline from sections 2b and 2c. You define the same pattern — a prompt template and grading criteria — and OpenAI handles execution, storage, and aggregation at scale.

We’ll use the `eval_dataset.json` generated in section 1e to run the same sentiment classification evaluation as a managed job, then compare results to what we computed in the DIY pipeline.

**Requires:** `OPENAI_API_KEY` in your `.env` file.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env in the working directory or any parent directory

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found. "
        "Add OPENAI_API_KEY=sk-... to your .env file and restart the kernel."
    )

# Separate client for OpenAI (distinct from the GitHub Models client above)
from openai import OpenAI as OpenAIClient
oai = OpenAIClient(api_key=OPENAI_API_KEY)
print("OpenAI client ready.")

#### Step 1 — Upload the dataset

The Evals API reads data from a JSONL file stored in OpenAI’s Files service. Each line is one test case: `{"input": "...", "expected": "..."}`.

In [ ]:
import json
import pathlib
import time

# Load the dataset generated in section 1e
dataset_path = pathlib.Path("eval_dataset.json")
eval_dataset_2d = json.loads(dataset_path.read_text())
print(f"Loaded {len(eval_dataset_2d)} items from {dataset_path}")

# Write as JSONL (one JSON object per line — required by OpenAI Files API)
jsonl_path = pathlib.Path("eval_dataset.jsonl")
with open(jsonl_path, "w") as f:
    for item in eval_dataset_2d:
        f.write(json.dumps(item) + "\n")

# Upload to OpenAI Files API with purpose="evals"
with open(jsonl_path, "rb") as f:
    uploaded_file = oai.files.create(file=f, purpose="evals")

print(f"File uploaded — ID: {uploaded_file.id}")
print(f"Status: {uploaded_file.status}")

In [ ]:
# --- Step 2: Create the eval definition ---
# The eval defines:
#   data_source_config — which model runs the prompt, and what template to use
#   testing_criteria   — how outputs are graded (string_check mirrors exact_match)

eval_def = oai.evals.create(
    name="anlp-session7-sentiment-eval",
    data_source_config={
        "type": "completions",
        "model": "gpt-4o-mini",
        "input_messages": {
            "type": "template",
            "template": [
                {
                    "role": "user",
                    "content": (
                        "Classify the sentiment as exactly one of: Positive, Negative, or Neutral.\n"
                        "Respond with only the label — no explanation.\n\n"
                        "Text: {{item.input}}\n"
                        "Sentiment:"
                    ),
                }
            ],
        },
        "source": {"type": "file_id", "id": uploaded_file.id},
    },
    testing_criteria=[
        {
            "type": "string_check",
            "name": "sentiment_label_match",
            # {{sample.output_text}} — the model's response to the prompt above
            "input": "{{sample.output_text}}",
            # ilike = case-insensitive substring match — same as exact_match() in section 2b
            "operation": "ilike",
            "reference": "{{item.expected}}",
        }
    ],
)

print(f"Eval created — ID: {eval_def.id}")

In [ ]:
# --- Step 3: Run the eval and wait for results ---
# The eval runs asynchronously. We poll until status changes from "running" to "completed".

print(f"Eval ID: {eval_def.id}")
print("Polling for results...")

MAX_WAIT_SECONDS = 180
poll_interval = 5
elapsed = 0

while elapsed < MAX_WAIT_SECONDS:
    current = oai.evals.retrieve(eval_def.id)
    status = getattr(current, "status", "unknown")
    print(f"  [{elapsed:3}s] status: {status}")

    if status == "completed":
        break
    if status in ("failed", "cancelled"):
        print(f"Eval ended with status: {status}")
        break

    time.sleep(poll_interval)
    elapsed += poll_interval
else:
    print("Timeout — the eval may still be running. Check the OpenAI dashboard.")

print(f"\nFinal status: {status}")

In [ ]:
# --- Step 4: Retrieve and display results ---
# Output items contain each test case, the model's output, and the grader result.

output_items = oai.evals.output_items.list(eval_def.id)

passed = 0
total = 0

print(f"{''  :>3} {'Expected':>10} {'Model output':>14}  Input")
print("-" * 80)

for item in output_items.data:
    # Navigate the output item structure
    expected = item.datasource_item.get("expected", "?")
    model_output = getattr(item.sample, "output", [{}])
    output_text = model_output[0].get("content", "?") if model_output else "?"
    # result is a list of grader results; first entry has the pass/fail
    result = item.results[0] if item.results else {}
    passed_flag = result.get("passed", False)

    icon = "\u2713" if passed_flag else "\u2717"
    passed += int(passed_flag)
    total += 1

    input_text = item.datasource_item.get("input", "")
    print(f"{icon:>3} {expected:>10} {output_text.strip():>14}  {input_text[:48]}...")

if total > 0:
    print(f"\nManaged eval accuracy: {passed/total:.1%}  ({passed}/{total})")
    print(f"DIY pipeline accuracy: {accuracy:.1%}  ({n_correct}/{len(eval_results)})")
    print("\nNote: Small differences are expected — the managed API runs independently of our DIY pipeline.")

**What to notice:**
- The managed eval uses the **same prompt template and grader logic** as the DIY pipeline — any difference in accuracy reflects model non-determinism, not a different approach.
- `string_check` with `ilike` is exactly `exact_match()` with case-insensitive substring matching.
- The Evals API stores results, enabling regression testing: re-run the same eval after updating your prompt to see whether accuracy improved.
- For production use, `label_model` replaces `llm_judge()` — same pattern, managed at scale.

| What | DIY (section 2b–2c) | OpenAI Evals API (this section) |
|---|---|---|
| Prompt template | Python f-string | Mustache `{{item.field}}` |
| Grading | `exact_match()` function | `string_check` grader |
| LLM judge | `llm_judge()` function | `label_model` grader |
| Scale | Manual loop | Managed, stored, version-controlled |

---

## Summary — Three Tiers of LLM Evaluation

| Tier | What | Tools | When to use |
|---|---|---|---|
| **A — Classical metrics** | Surface/semantic overlap | BLEU, ROUGE, BERTScore, perplexity | Translation, summarisation benchmarks |
| **B — Eval tools** | Prompt testing at scale | Promptfoo, OpenAI Evals API | Compare prompts/models, regression testing |
| **C — Custom pipelines** | Task-specific scoring | DIY eval loop, LLM-as-judge | Your real project — AT2 and beyond |

**Which tier for your AT2 project?**
- If your task has a reference answer (translation, summarisation): start with ROUGE + BERTScore
- If your task is classification: use `exact_match` with your own test cases
- If your task is open-ended (QA, generation): use LLM-as-judge with a well-defined rubric
- Combine tiers — classical metrics for speed, LLM-as-judge for quality assessment

**Remember:** Benchmark scores tell you about general capability. Your own eval tells you about your task. The further your task is from standard benchmarks, the more important your own eval becomes.